In [1]:
#!/usr/bin/env python
#-*- coding: utf-8-*
"""
Student Performance Data Analysis Script
This script loads student performance data, cleans it, and generates
visualizations to understand factors affecting final exam scores.
Author: [Muhammad Ayyan]
Date: [01-04-2026]
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import argparse
import os
def load_data(filepath):
    """
    Load student performance data from CSV file.
    Parameters:----------
    filepath : str
    Path to the CSV file
    Returns:--------
    pandas.DataFrame
    Loaded dataframe
    """
    print(f"Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
    return df
def clean_data(df):
    """
    Clean the dataframe by handling missing values.
    Parameters:----------
    df : pandas.DataFrame
    Raw dataframe
    Returns:--------
    pandas.DataFrame
    Cleaned dataframe
    """
    print("Cleaning data...")
    df_clean = df.copy()
    # Fill missing study_hours with median
    if df_clean['study_hours'].isnull().any():
        median_study = df_clean['study_hours'].median()
        df_clean['study_hours'] = df_clean['study_hours'].fillna(median_study)
        print(f"- Filled {df_clean['study_hours'].isnull().sum()} missing study_hours with median ({median_study:.2f})")
    # Fill missing sleep_hours with mean
    if df_clean['sleep_hours'].isnull().any():
        mean_sleep = df_clean['sleep_hours'].mean()
        df_clean['sleep_hours'] = df_clean['sleep_hours'].fillna(mean_sleep)
        print(f"- Filled {df_clean['sleep_hours'].isnull().sum()} missing sleep_hours with mean ({mean_sleep:.2f})")
    print(f"Cleaning complete. {df_clean.isnull().sum().sum()} missing values remain.")
    return df_clean
def generate_summary_statistics(df, output_dir='output'):
    """
    Generate and save summary statistics.
    Parameters:----------
    df : pandas.DataFrame
    Cleaned dataframe
    output_dir : str
    Directory to save output files
    """
    print("Generating summary statistics...")
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    # Summary statistics
    summary = df.describe()
    summary.to_csv(f'{output_dir}/summary_statistics.csv')
    print(f"- Saved summary statistics to {output_dir}/summary_statistics.csv")
    # Missing values report
    missing = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percent': (df.isnull().sum() / len(df)) * 100
    })
    missing.to_csv(f'{output_dir}/missing_values.csv', index=False)
    print(f"- Saved missing values report to {output_dir}/missing_values.csv")
    return summary, missing
def create_visualizations(df, output_dir='output'):
    """
    Create and save visualization plots.
    Parameters:----------
    df : pandas.DataFrame
    Cleaned dataframe
    output_dir : str
    Directory to save output files
    """
    print("Creating visualizations...")
    os.makedirs(f'{output_dir}/plots', exist_ok=True)
    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    # 1. Distribution plots
    numerical_cols = ['study_hours', 'sleep_hours', 'prev_exam_score',
    'attendance', 'final_score']
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for i, col in enumerate(numerical_cols):
        row = i // 3
        col_idx = i % 3
        axes[row, col_idx].hist(df[col], bins=20, edgecolor='black', alpha=0.7)
        axes[row, col_idx].set_title(f'Distribution of {col}')
        axes[row, col_idx].set_xlabel(col)
        axes[row, col_idx].set_ylabel('Frequency')
    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/plots/distributions.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"- Saved distributions plot")
    # 2. Correlation heatmap
    plt.figure(figsize=(8, 6))
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
    square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"- Saved correlation heatmap")
    # 3. Scatter plots matrix (simplified)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    features = ['study_hours', 'sleep_hours', 'prev_exam_score', 'attendance']
    for i, feature in enumerate(features):
        row = i // 2
        col = i % 2
        scatter = axes[row, col].scatter(
        df[feature],
        df['final_score'],
        alpha=0.6,
        c=df['final_score'],
        cmap='viridis',
        edgecolor='black',
        linewidth=0.5)
        axes[row, col].set_xlabel(feature)
        axes[row, col].set_ylabel('Final Score')
        axes[row, col].set_title(f'{feature} vs Final Score')
        axes[row, col].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/plots/scatter_plots.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"- Saved scatter plots")
    # 4. Box plots by category
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # By gender
    df.boxplot(column='final_score', by='gender', ax=axes[0])
    axes[0].set_title('Final Score by Gender')
    axes[0].set_xlabel('Gender')
    axes[0].set_ylabel('Final Score')
    # By major
    df.boxplot(column='final_score', by='major', ax=axes[1])
    axes[1].set_title('Final Score by Major')
    axes[1].set_xlabel('Major')
    axes[1].set_ylabel('Final Score')
    plt.suptitle('Final Score by Category')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/plots/categorical_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"- Saved categorical analysis plot")
    print(f"All visualizations saved to {output_dir}/plots/")
def main():
    """
    Main function to orchestrate the analysis.
    """
    # Parse command line arguments
    parser = argparse.ArgumentParser(description='Analyze student performance data')
    parser.add_argument('--input', type=str, default='student_performance.csv',
    help='Input CSV file path')
    parser.add_argument('--output', type=str, default='output',
    help='Output directory for results')
    args = parser.parse_args([])
    args.input = "student_performance.csv"
    args.output = "output"
    print("=" * 50)
    print("STUDENT PERFORMANCE DATA ANALYSIS")
    print("=" * 50)
    # Check if input file exists
    if not os.path.exists(args.input):
        print(f"Error: Input file '{args.input}' not found.")
        return
    # Execute analysis pipeline
    df_raw = load_data(args.input)
    df_clean = clean_data(df_raw)
    generate_summary_statistics(df_clean, args.output)
    create_visualizations(df_clean, args.output)
    print("\n" + "=" * 50)
    print("ANALYSIS COMPLETE!")
    print(f"Results saved to '{args.output}' directory")
    print("=" * 50)
if __name__ == "__main__":
    main()

STUDENT PERFORMANCE DATA ANALYSIS
Loading data from student_performance.csv...
Loaded 200 rows and 8 columns
Cleaning data...
- Filled 0 missing study_hours with median (9.62)
- Filled 0 missing sleep_hours with mean (7.05)
Cleaning complete. 0 missing values remain.
Generating summary statistics...
- Saved summary statistics to output/summary_statistics.csv
- Saved missing values report to output/missing_values.csv
Creating visualizations...
- Saved distributions plot
- Saved correlation heatmap
- Saved scatter plots
- Saved categorical analysis plot
All visualizations saved to output/plots/

ANALYSIS COMPLETE!
Results saved to 'output' directory
